# RLHF full evaluation and curation notebook

Use this notebook to browse the final 1024-token policy-suite outputs and manually review qualified improvements, ordinary failures, and reward-model mismatches. It is intentionally interactive and centered on example inspection.

In [ ]:
from pathlib import Path
import os, sys, re
import pandas as pd
from IPython.display import display, Markdown, HTML

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EVAL_DIR = Path("outputs/rlhf/qwen25_05b_helpsteer3_eval_suite_4096_ep2_u400_eval1024")
SAMPLES = EVAL_DIR / "policy_suite_samples.csv"
df = pd.read_csv(SAMPLES)
policies = ["base", "sft_4096", "ppo_4096_ep2_u400"]
print("Loaded", len(df), "examples from", SAMPLES)

In [ ]:
def extract_last_user_prompt(rendered_prompt: str) -> str:
    matches = re.findall(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", str(rendered_prompt), flags=re.S)
    if matches:
        return matches[-1].strip()
    return str(rendered_prompt).replace("<|im_start|>assistant\n", "").strip()


def compact_text(text, max_chars=1800):
    text = str(text).strip()
    if max_chars is not None and len(text) > max_chars:
        return text[:max_chars] + "\n\n...[truncated for display]"
    return text


def show_example(idx: int, max_chars=2500):
    row = df[df["idx"] == idx]
    if row.empty:
        raise ValueError(f"idx {idx} not found")
    r = row.iloc[0]
    display(Markdown(f"# idx {idx} — {r['domain']} / {r['language']}"))
    display(Markdown(f"**winner:** `{r['winner']}`  \n**reward rank:** `{r['reward_rank']}`  \n**spread:** `{r['reward_spread']:.4f}`"))
    display(Markdown("## Prompt\n" + extract_last_user_prompt(r["prompt"])))
    for p in policies:
        display(Markdown(
            f"## {p}\n"
            f"reward: `{r[f'{p}_reward']:.4f}` | "
            f"tokens: `{int(r[f'{p}_response_tokens'])}` | "
            f"cap_hit: `{bool(r[f'{p}_cap_hit'])}` | "
            f"hit_eos: `{bool(r[f'{p}_hit_eos'])}`\n\n"
            + compact_text(r[f"{p}_response"], max_chars=max_chars)
        ))


def table_for(indices):
    cols = [
        "idx", "domain", "language", "winner", "reward_rank", "reward_spread",
        "base_reward", "sft_4096_reward", "ppo_4096_ep2_u400_reward",
        "delta_sft_4096_minus_base",
        "delta_ppo_4096_ep2_u400_minus_base",
        "delta_ppo_4096_ep2_u400_minus_sft_4096",
        "base_response_tokens", "sft_4096_response_tokens", "ppo_4096_ep2_u400_response_tokens",
    ]
    return df[df.idx.isin(indices)][cols].sort_values("idx")

## Candidate pools

In [ ]:
strong_ppo_wins = df[
    (df["winner"] == "ppo_4096_ep2_u400") &
    (df["delta_ppo_4096_ep2_u400_minus_base"] > 2.0) &
    (df["delta_ppo_4096_ep2_u400_minus_sft_4096"] > 1.0)
].sort_values("delta_ppo_4096_ep2_u400_minus_base", ascending=False)

sft_wins = df[
    (df["winner_base_vs_sft_4096"] == "sft_4096") &
    (df["delta_sft_4096_minus_base"] > 2.0)
].sort_values("delta_sft_4096_minus_base", ascending=False)

base_wins = df[
    (df["winner_base_vs_ppo_4096_ep2_u400"] == "base") &
    (df["delta_ppo_4096_ep2_u400_minus_base"] < -5.0)
].sort_values("delta_ppo_4096_ep2_u400_minus_base")

ties = df[df["winner"] == "tie"].copy()

print("strong PPO wins:", len(strong_ppo_wins))
print("strong SFT wins:", len(sft_wins))
print("strong PPO losses to base:", len(base_wins))
print("ties:", len(ties))

display(strong_ppo_wins[["idx", "domain", "language", "reward_rank", "delta_ppo_4096_ep2_u400_minus_base", "delta_ppo_4096_ep2_u400_minus_sft_4096"]].head(30))

## Inspect one example by index

Change the index and rerun the cell.

In [ ]:
show_example(0, max_chars=3000)

## Browse a category

In [ ]:
# Change these values to browse different slices.
category = "strong_ppo_wins"  # strong_ppo_wins, sft_wins, base_wins, ties
rank = 0

pools = {
    "strong_ppo_wins": strong_ppo_wins,
    "sft_wins": sft_wins,
    "base_wins": base_wins,
    "ties": ties,
}
idx = int(pools[category].iloc[rank]["idx"])
print("Showing", category, "rank", rank, "idx", idx)
show_example(idx, max_chars=3000)

## Save a manually curated set

In [ ]:
# Curated 1024-token examples: qualified improvements, failures, and reward-model misses.
selected_indices = [2, 0, 1210, 1158, 418, 1511, 1970, 1303, 176, 1835, 1548, 346, 1281, 1579, 656, 86]

def export_selected(indices, path=EVAL_DIR / "selected_qualitative_examples.md", max_chars=5000):
    lines = ["# Selected RLHF qualitative examples", ""]
    lines.append("These examples were selected from the 1024-token policy-suite evaluation to show both improvements and failure modes.")
    lines.append("")
    for idx in indices:
        r = df[df.idx == idx].iloc[0]
        lines.append(f"## idx {idx} — {r['domain']} / {r['language']} — winner `{r['winner']}`")
        lines.append("")
        lines.append("### Prompt")
        lines.append(extract_last_user_prompt(r["prompt"]))
        lines.append("")
        for p in policies:
            text = compact_text(r[f"{p}_response"], max_chars=max_chars)
            lines.append(f"### {p} — reward `{r[f'{p}_reward']:.4f}`, tokens `{int(r[f'{p}_response_tokens'])}`")
            lines.append("")
            lines.append(text)
            lines.append("")
        lines.append("---")
        lines.append("")
    path = Path(path)
    path.write_text("\n".join(lines), encoding="utf-8")
    return path

out = export_selected(selected_indices)
print("Wrote", out)

## Optional: export all candidate pools to CSV

In [ ]:
strong_ppo_wins.to_csv(EVAL_DIR / "curation_strong_ppo_wins.csv", index=False)
sft_wins.to_csv(EVAL_DIR / "curation_strong_sft_wins.csv", index=False)
base_wins.to_csv(EVAL_DIR / "curation_strong_base_wins.csv", index=False)
ties.to_csv(EVAL_DIR / "curation_ties.csv", index=False)
print("Wrote curation CSVs to", EVAL_DIR)